## Queries - LSSTCam

In this notebook, we show how to query the LSSTCam repository\
and view the resulting images.\

Craig Lage - 15-Apr_25

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from lsst.daf.butler import Butler
from lsst.ip.isr import IsrTask, IsrTaskConfig
from lsst.summit.utils.plotting import plot
import lsst.afw.cameraGeom.utils as camGeomUtils
import lsst.summit.utils.butlerUtils as butlerUtils
import lsst.afw.math as afwMath
import lsst.afw.display as afwDisplay
import lsst.geom as geom
from lsst.geom import Point2D
from lsst.obs.lsst import LsstCam
from PIL import Image
import os

In [ ]:
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)

In [ ]:
camera = LsstCam.getCamera()
detector = camera['R22_S11']
bbox = detector.getBBox()

expId = 2026031000072
detector =94
instrument = 'LSSTCam'
calexp72 = butler.get('preliminary_visit_image', detector=detector, visit=expId, instrument=instrument)
wcs72 = calexp72.getWcs()
calExpSkyCenter = wcs72.pixelToSky(Point2D(bbox.centerX, bbox.centerY))
ra_72 = calExpSkyCenter.getRa().asDegrees()
dec_72 = calExpSkyCenter.getDec().asDegrees()

fig = plt.figure(figsize=(10,10))
x = plot(calexp72.image.array[1950:2050, 1998:2098], figure=fig, stretch='ccs')
ax = x.get_axes()[0]
ax.set_title(f"{expId}_{detector}\nRA_center={ra_72:.6f}, Dec_center={dec_72:.6f}\ndRA={(ra_72-ra_72)*3600.0:.2f} arcsec  dDec={(dec_72-dec_72)*3600.0:.2f} arcsec")
ax.text(50,50, "+", color='magenta', fontsize=64, ha='center', va='center')
ax.axvline(25, color='red')
ax.axhline(34, color='red')
image_72 = f"/home/c/cslage/u/LSSTCam/images/WCS_{expId}.png"
fig.savefig(image_72)

expId = 2026031000073
detector =94
instrument = 'LSSTCam'
calexp73 = butler.get('preliminary_visit_image', detector=detector, visit=expId, instrument=instrument)
wcs73 = calexp73.getWcs()
wcs72 = calexp72.getWcs()
calExpSkyCenter = wcs73.pixelToSky(Point2D(bbox.centerX, bbox.centerY))
ra_73 = calExpSkyCenter.getRa().asDegrees()
dec_73 = calExpSkyCenter.getDec().asDegrees() 

fig = plt.figure(figsize=(10,10))
x = plot(calexp73.image.array[1950:2050, 1998:2098], figure=fig, stretch='ccs')
ax = x.get_axes()[0]
ax.set_title(f"{expId}_{detector}\nRA_center={ra_72:.6f}, Dec_center={dec_72:.6f}\ndRA={(ra_73-ra_72)*3600.0:.2f} arcsec  dDec={(dec_73-dec_72)*3600.0:.2f} arcsec")
ax.text(50,50, "+", color='magenta', fontsize=64, ha='center', va='center')
ax.axvline(25, color='red')
ax.axhline(34, color='red')
image_73 = f"/home/c/cslage/u/LSSTCam/images/WCS_{expId}.png"
fig.savefig(image_73)

In [ ]:
def merge_images_side_by_side(img1_path, img2_path, output_path="merged_image.png"):
    """
    Merges two PNG images side by side. Handles different heights by 
    resizing the smaller image proportionally to match the taller one.
    Retains transparency by using RGBA mode.
    """
    # Open the images
    img1 = Image.open(img1_path).convert("RGBA")
    img2 = Image.open(img2_path).convert("RGBA")

    # Determine the dimensions for the merged image
    # The new image height will be the maximum height of the two images
    # The new image width will be the sum of the widths of the two images
    max_height = max(img1.height, img2.height)
    total_width = img1.width + img2.width

    # Resize images to a common height to avoid misalignment
    if img1.height != max_height:
        # Calculate new width to maintain aspect ratio
        new_width1 = int(img1.width * max_height / img1.height)
        img1 = img1.resize((new_width1, max_height), Image.Resampling.LANCZOS)
        
    if img2.height != max_height:
        new_width2 = int(img2.width * max_height / img2.height)
        img2 = img2.resize((new_width2, max_height), Image.Resampling.LANCZOS)

    # Create a new blank image with the calculated total size and RGBA mode
    # The color (0, 0, 0, 0) makes the background transparent
    merged_image = Image.new('RGBA', (total_width, max_height), (0, 0, 0, 0))

    # Paste the first image at the top-left corner (0, 0)
    merged_image.paste(img1, (0, 0))
    
    # Paste the second image next to the first one, using the first image's width as the x-offset
    # The total width is used here as the new x-offset, but since we resized, we must use 
    # the new width of img1 for accurate placement
    merged_image.paste(img2, (img1.width, 0))

    # Save the resulting image as a PNG file
    merged_image.save(output_path, format="PNG")
    print(f"Images merged and saved to {os.path.abspath(output_path)}")



In [ ]:
# Example usage:
# Replace 'image1.png' and 'image2.png' with your file paths
merge_images_side_by_side(image_72, image_73, "/home/c/cslage/u/LSSTCam/images/WCS_72-73.png")